In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector, StackingCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier, FTTransformerHead
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
# PandasConverter converts to pandas, setting 'id' as the index.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr)
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_org = loader.transform([data_path / 'WA_Fn-UseC_-Telco-Customer-Churn.csv']).drop('customerID').with_columns(
    id=-pl.int_range(1, pl.len() + 1),
    tenure=pl.col('tenure').clip(1),
    Churn=(pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

2026-03-15 21:16:30.695545: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-15 21:16:30.733261: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-15 21:16:31.484796: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

3.12.12 (main, Dec 27 2025, 11:08:36) [GCC 13.3.0]
pandas 2.3.3
polars 1.38.1
numpy 2.3.5
matplotlib.pyplot
seaborn 0.13.2
scipy 1.16.3
sklearn 1.8.0
tensorflow 2.20.0
xgboost 3.2.0
lightgbm
catboost 1.2.10
mllabs 0.6.3


In [2]:
if os.path.exists('exp/skf5_aug'):
    e_skf5 = Experimenter.load('exp/skf5_aug', df_train, aug_data = df_org)
    if e_skf5.status == 'closed':
        e_skf5.reopen_exp()
else:
    e_skf5 = Experimenter.create(
        df_train, 'exp/skf5_aug', title='The experimentation using 5-fold StratifiedKFold',
        sp=StratifiedKFold(n_splits=5, shuffle=True, random_state=1),
        splitter_params={'y': target}, aug_data = df_org
    )

e_skf5.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_skf5.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)
e_skf5.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges=y_edges), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_skf5.add_collector(
    StackingCollector(
        'stacking', Connector(edges=y_edges),
        slice(-1, None), method='mean', experimenter = e_skf5
    )
)
e_skf5.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01
    }
)

e_skf5.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)

e_skf5.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': 0, 'random_state': 1, 'max_depth': 5, 'colsample_bylevel': 0.75
    }
)

## Neural network
e_skf5.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 0, 'validation_fraction': 0})

Markdown(
    e_skf5.desc_spec()
)

Loaded: 54 node(s), 6 group(s), 5 fold(s)


## The experimentation using 5-fold StratifiedKFold

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=5, random_state=1, shuffle=True)` |
| **Inner Splitter (sp_v)** | None |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 5 |
| **Inner Folds** | 1 |

In [3]:
from sklearn.preprocessing import KBinsDiscretizer, RobustScaler, PolynomialFeatures, StandardScaler, TargetEncoder, MinMaxScaler
from mllabs.processor import FrequencyEncoder, CatConverter, CatOOVFilter, TypeConverter
from sklearn.impute import SimpleImputer
e_skf5.set_node(
    'si', grp='pre', processor=SimpleImputer, edges = {'X': [(None, X_num)]}
)
e_skf5.set_node(
    'kbin', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', None)]}, params = {'n_bins': 10, 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_skf5.set_node(
    'rb', grp='pre', processor=RobustScaler, edges = {'X': [('si', None)]}, params = {}
)
e_skf5.set_node(
    'std', grp='pre', processor=StandardScaler, edges = {'X': [('si', None)]}, params = {}
)
e_skf5.set_node(
    'tgt_num', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('si', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_skf5.set_node(
    'freq_num', grp='pre', processor=FrequencyEncoder, method = 'transform', edges = {'X': [('si', None)], **y_edges}, params = {'normalize': True}
)
e_skf5.set_node(
    'n2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [('si', None)]}, params={'to': 'str'}
)

e_skf5.set_node(
    'n2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('n2s', None), ('kbin', None)]}
)

e_skf5.set_node(
    'coov', grp='pre', processor=CatOOVFilter, method = 'transform', edges = {'X': [('n2c', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)

e_skf5.set_node(
    'rb_std', grp='pre', processor=RobustScaler, edges = {'X': [('rb', None)]}, params = {}
)
e_skf5.set_node(
    'mm', grp='pre', processor=MinMaxScaler, edges = {'X': [(None, X_num)]}, params = {}
)
e_skf5.set_node(
    'expr3', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
    }, 'with_columns': False}
)
e_skf5.set_node(
    'expr4', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
        'Total_per_Monthly_ratio': pl.col('si__TotalCharges') / pl.col('si__MonthlyCharges'),
        'TotalCharges_per_tenure_min': pl.min_horizontal(pl.col('si__TotalCharges') / pl.col('si__tenure'), pl.col('si__MonthlyCharges')),
        'TotalCharges_per_tenure_max': pl.max_horizontal(pl.col('si__TotalCharges') / pl.col('si__tenure'), pl.col('si__MonthlyCharges')),
    }, 'with_columns': False}
)
service = ['DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 'StreamingTV_Y', 'TechSupport_Y']
e_skf5.set_node(
    'expr5', grp='pre', processor = ExprProcessor, edges = {'X': [(None, service)]},
    params={'dict_expr': {
        'Service_cnt':  pl.sum_horizontal(service),
    }, 'with_columns': False}
)
e_skf5.set_node(
    'expr3_std', grp='pre', processor = StandardScaler, edges = {'X': [('expr3', None)]}
)
e_skf5.set_node(
    'expr4_std', grp='pre', processor = StandardScaler, edges = {'X': [('expr4', None)]}
)
e_skf5.build()

Building 0 node(s)
Build 5/5 (100%)        
Build complete: 0 node(s)


In [4]:
from mllabs.processor import CatPairCombiner
from itertools import combinations

TOP_CATS_FOR_NGRAM = ['Contract', 'DSL_Y', 'PaymentMethod', 'OnlineSecurity']

bigram_cols = [(i1, i2) for i1, i2 in combinations(TOP_CATS_FOR_NGRAM, 2)]
trigram_cols = [(i1, i2, i3) for i1, i2, i3 in combinations(TOP_CATS_FOR_NGRAM, 3)]

e_skf5.set_node(
    'bigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM)]}, params={'pairs': bigram_cols}
)
e_skf5.set_node(
    'trigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM[:4])]}, params={'pairs': trigram_cols}
)
e_skf5.set_node(
    'tgt_bigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('bigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_skf5.set_node(
    'tgt_trigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('trigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_skf5.set_node(
    'coov_bigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('bigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_skf5.set_node(
    'coov_trigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('trigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_skf5.build()

Building 0 node(s)
Build 5/5 (100%)        
Build complete: 0 node(s)


In [5]:
params = [(3, 2500, 0.05), (4, 1400, 0.05), (5, 2000, 0.025)]
for i, (max_depth, n_estimators, learning_rate) in enumerate(params):
    e_skf5.set_node(
        'xgb_{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'max_depth': max_depth, 'n_estimators': n_estimators, 'learning_rate': learning_rate}
    )
e_skf5.exp()

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


In [45]:
params = [(7, 2500, 0.05), (15, 2100, 0.025)]
for i, (nl, n_estimators, learning_rate) in enumerate(params):
    e_skf5.set_node(
        'lgb_{}'.format(i), grp = 'lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'num_leaves': nl, 'n_estimators': n_estimators, 'learning_rate': learning_rate}
    )
e_skf5.exp()

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


In [62]:
e_skf5.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False).iloc[:10]

,valid,train_sub
xgb_6,0.918300,0.921197
lgb_5,0.918284,0.921437
lgb_6,0.918265,0.921637
xgb_5,0.918174,0.920860
cb_4,0.918126,0.924402
lgb_4,0.918097,0.921059
cb_1,0.918042,0.924072
xgb_4,0.916701,0.918077
xgb_0,0.916676,0.917965
lgb_3,0.916660,0.918201


In [47]:
i = 3
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None)]}, 
    params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


In [48]:
e_skf5.get_collector('AUC').get_metrics_agg('lgb_3')[0].sort_values('valid', ascending = False)

,valid,train_sub
lgb_3,0.91666,0.918201


In [49]:
i = 4
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None)]}, 
    params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


,valid,train_sub
lgb_4,0.918097,0.921059


In [50]:
i = 5
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr3', None)]}, 
    params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


,valid,train_sub
lgb_5,0.918284,0.921437


In [64]:
i = 6
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None)]
    }, params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
lgb_6,0.918314,0.921606


In [65]:
i = 7
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None)]
    }, params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
lgb_7,0.918273,0.921648


In [5]:
i = 8
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('tgt_bigram', None)]
    }, params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
lgb_8,0.918275,0.921882


In [11]:
i = 9
e_skf5.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'num_leaves': 7, 'n_estimators': 2500, 'learning_rate': 0.05, 'colsample_bytree':0.5}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)

Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
lgb_9,0.918311,0.921743


In [51]:
i = 4
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None)]}, 
    params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


In [52]:
e_skf5.get_collector('AUC').get_metrics_agg('xgb_4')[0].sort_values('valid', ascending = False)

,valid,train_sub
xgb_4,0.916701,0.918077


In [16]:
i = 5
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None)]}, 
    params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
xgb_5,0.918174,0.92086


In [17]:
i = 6
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr3', None)]}, 
    params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
xgb_6,0.9183,0.921197


In [66]:
i = 7
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None)]
    }, params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
xgb_7,0.918237,0.921426


In [6]:
i = 8
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr3', None), ('tgt_bigram', None)]
    }, params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
xgb_8,0.918271,0.921496


In [8]:
i = 9
e_skf5.set_node(
    'xgb_{}'.format(i), grp = 'xgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'max_depth': 3, 'n_estimators': 2500, 'learning_rate': 0.05, 'colsample_bytree': 0.5}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('xgb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                                   
Experimentation complete: 1 node(s)


,valid,train_sub
xgb_9,0.918311,0.921613


In [18]:
i = 1
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None)]}, 
    params={'colsample_bylevel':0.9, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


,valid,train_sub
cb_1,0.918042,0.924072


In [19]:
i = 2
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None)]}, 
    params={'colsample_bylevel':0.9, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


,valid,train_sub
cb_2,0.916567,0.919746


In [20]:
i = 4
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr3', None)]}, 
    params={'colsample_bylevel':0.9, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


,valid,train_sub
cb_4,0.918126,0.924402


In [16]:
i = 5
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'colsample_bylevel':0.5, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('cb_{}'.format(i))[0].sort_values('valid', ascending = False)

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


,valid,train_sub
cb_5,0.918274,0.925528


In [26]:
e_skf5.set_node(
    'nn_0', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [256, 128]}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 20}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 0/5 (0%) > nn_0 0/1 (0%)

I0000 00:00:1773504972.503044  219657 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5488 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


In [ ]:
e_skf5.set_node(
    'nn_0', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [256, 128]}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 20}
)
e_skf5.exp()

In [4]:
e_skf5.set_node(
    'nn_1', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [256, 128]}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 20, 'learning_rate':0.0005}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 0/5 (0%) > nn_1 0/1 (0%)

I0000 00:00:1773545531.037728    1211 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5518 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


In [5]:
e_skf5.set_node(
    'nn_2', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [128, 64]}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 20, 'learning_rate':0.0005}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


In [6]:
e_skf5.set_node(
    'nn_3', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [64, 32]}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 20, 'learning_rate':0.0005}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


In [21]:
e_skf5.set_node(
    'nn_4', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [64, 32]}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 15, 'learning_rate':0.0002}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


In [23]:
e_skf5.set_node(
    'nn_5', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [64, 32]}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 10, 'learning_rate':0.0001}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


In [47]:
e_skf5.set_node(
    'nn_6', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None), ('tgt_bigram', None), ('tgt_trigram', None)]},
    params={'hidden': {'units': [64, 32], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 10, 'learning_rate':0.0001}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('nn_6')[0].sort_values('valid', ascending = False).iloc[:10]

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


,valid,train_sub
nn_6,0.914637,0.92405


In [48]:
e_skf5.set_node(
    'nn_7', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr4_std', None), ('coov_bigram', None), ('coov_trigram', None)]},
    params={'hidden': {'units': [64, 32], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 10, 'learning_rate':0.0001}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('nn_7')[0].sort_values('valid', ascending = False).iloc[:10]

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


,valid,train_sub
nn_7,0.914949,0.924419


In [59]:
e_skf5.set_node(
    'nn_8', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr4_std', None), 
                 ('tgt_bigram', None), ('tgt_trigram', None), ('coov_bigram', None), ('coov_trigram', None)]},
    params={'hidden': {'units': [64, 32], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 10, 'learning_rate':0.0001}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('nn_8')[0].sort_values('valid', ascending = False).iloc[:10]

Experimenting 1 node(s)
Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


,valid,train_sub
nn_8,0.915005,0.924388


In [36]:
e_skf5.set_node(
    'nn_9', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('expr4_std', None), ('coov_bigram', None), ('coov_trigram', None)]},
    params={'head': FTTransformerHead, 'head_params': {'d_model': 8, 'n_heads':2, 'n_layers': 2}, 'hidden': {'units': [32, 16], 'dropout': 0.5}, 
            'cat_cols': ColSelector(col_type = 'category'), 'epochs': 10, 'learning_rate':0.0001}
)
e_skf5.exp()
e_skf5.get_collector('AUC').get_metrics_agg('nn_9')[0].sort_values('valid', ascending = False).iloc[:10]

Experimenting 1 node(s)
Exp 5/5 (100%)                                                        
Experimentation complete: 1 node(s)


,valid,train_sub
nn_9,0.914538,0.922892


In [20]:
e_skf5.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False).iloc[:10]

,valid,train_sub
lgb_6,0.918314,0.921606
lgb_9,0.918311,0.921743
xgb_9,0.918311,0.921613
xgb_6,0.918300,0.921197
lgb_5,0.918284,0.921437
lgb_8,0.918275,0.921882
cb_5,0.918274,0.925528
lgb_7,0.918273,0.921648
xgb_8,0.918271,0.921496
xgb_7,0.918237,0.921426


# Stacking

In [37]:
df_stacking = e_skf5.get_collector('stacking').get_dataset()
df_stacking.head()

xgb_0__Churn_1,lgb_5__Churn_1,nn_9__Churn_1,xgb_4__Churn_1,cb_2__Churn_1,lgb_8__Churn_1,lgb_4__Churn_1,nn_7__Churn_1,xgb_6__Churn_1,xgb_1__Churn_1,lgb_9__Churn_1,lgb_7__Churn_1,xgb_7__Churn_1,nn_1__Churn_1,nn_8__Churn_1,lgb_6__Churn_1,nn_6__Churn_1,lgb_0__Churn_1,xgb_5__Churn_1,cb_5__Churn_1,xgb_8__Churn_1,nn_2__Churn_1,xgb_2__Churn_1,xgb_9__Churn_1,nn_4__Churn_1,nn_0__Churn_1,nn_3__Churn_1,nn_5__Churn_1,lgb_3__Churn_1,lgb_1__Churn_1,cb_4__Churn_1,cb_1__Churn_1,Churn
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
0.011443,0.010374,0.004864,0.010329,0.010638,0.008045,0.01105,0.009153,0.012299,0.009359,0.009985,0.010836,0.011357,0.000039,0.008472,0.010657,0.011506,0.010294,0.011691,0.009062,0.011193,0.002088,0.012583,0.011835,0.010325,0.000289,0.01033,0.015504,0.011235,0.009974,0.01288,0.00993,0
0.012354,0.013297,0.001456,0.011476,0.011574,0.011563,0.012891,0.006818,0.0111,0.011651,0.008563,0.01077,0.011171,8.0833e-12,0.007282,0.010076,0.004549,0.01592,0.019452,0.020016,0.019708,0.000006,0.014198,0.020997,0.003873,7.5103e-15,0.000002,0.007652,0.018026,0.014925,0.013527,0.014482,0
0.959053,0.961993,0.873389,0.959201,0.965642,0.959385,0.963483,0.941536,0.967282,0.959089,0.959804,0.966284,0.966838,0.998214,0.954962,0.964882,0.953615,0.959903,0.962628,0.964992,0.966509,0.998381,0.958652,0.964658,0.971946,0.997016,0.984642,0.949319,0.959724,0.957823,0.967947,0.96701,1
0.639111,0.549537,0.573244,0.622238,0.634243,0.539066,0.567815,0.523853,0.547403,0.619764,0.555309,0.576072,0.565522,0.49174,0.548588,0.538383,0.538389,0.625891,0.59211,0.555898,0.544985,0.479155,0.613913,0.535669,0.594873,0.448994,0.539484,0.550452,0.608325,0.61656,0.527256,0.582073,0
0.191891,0.207636,0.257046,0.192358,0.168677,0.213603,0.185293,0.17391,0.211487,0.191628,0.227611,0.237198,0.25269,0.131479,0.179768,0.218167,0.264728,0.184903,0.218014,0.19571,0.201886,0.068449,0.177014,0.224144,0.242402,0.133459,0.123562,0.259143,0.190581,0.190578,0.142524,0.185368,0


In [38]:
if os.path.exists('exp/stacking_aug'):
    e_stacking = Experimenter.load('exp/stacking_aug', df_stacking)
    if e_stacking.status == 'closed':
        e_stacking.reopen_exp()
else:
    e_stacking = Experimenter.create(
        df_stacking, 'exp/stacking_aug', title='Stacking',
        sp=StratifiedKFold(n_splits=5, random_state=1, shuffle=True),
        splitter_params={'y': target}
    )

e_stacking.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges={'y': [(None, target)]}),
        slice(-1, None),
        roc_auc_score,
        include_train = True
    )
)

e_stacking.set_grp('pre', role='stage', method='transform')
e_stacking.build()
e_stacking.set_grp(
    'clf', role='head', method='predict_proba',
    edges={'y': [(None, target)]}
)
e_stacking.set_grp('lr', parent='clf', processor = LogisticRegression)

Loaded: 12 node(s), 3 group(s), 5 fold(s)
Building 0 node(s)
Build 5/5 (100%)        
Build complete: 0 node(s)


{'result': 'skip',
 'grp': <mllabs._pipeline.PipelineGroup at 0x7ddb716b7f80>,
 'affected_nodes': []}

In [39]:
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_1', 'xgb_2']]
e_stacking.set_node('lr1', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2']]
e_stacking.set_node('lr2', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'xgb_2']]
e_stacking.set_node('lr3', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2', 'lgb_0']]
e_stacking.set_node('lr4', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'xgb_2', 'lgb_0', 'lgb_1']]
e_stacking.set_node('lr5', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'lgb_0']]
e_stacking.set_node('lr6', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['lgb_3', 'xgb_4']]
e_stacking.set_node('lr15', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_2', 'lgb_3', 'xgb_4']]
e_stacking.set_node('lr16', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['cb_1', 'xgb_5', 'lgb_4', 'lgb_3', 'xgb_4']]
e_stacking.set_node('lr17', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['cb_4', 'cb_1', 'lgb_3', 'xgb_4', 'xgb_6', 'lgb_5', 'nn_6']]
e_stacking.set_node('lr18', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['cb_4', 'cb_5', 'lgb_3', 'xgb_4', 'xgb_6', 'lgb_5', 'nn_7', 'lgb_8', 'xgb_9']]
e_stacking.set_node('lr19', grp = 'lr', edges = {'X': [(None, X_sel)]})
e_stacking.exp()

Experimenting 0 node(s)
Exp 5/5 (100%)        
Experimentation complete: 0 node(s)


In [42]:
X_sel = ['{}__Churn_1'.format(i) for i in ['cb_4', 'cb_5', 'lgb_3', 'xgb_4', 'xgb_6', 'lgb_5', 'nn_8', 'lgb_8', 'xgb_9']]
e_stacking.set_node('lr20', grp = 'lr', edges = {'X': [(None, X_sel)]})
e_stacking.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


In [26]:
from mllabs.collector import ModelAttrCollector
coef = ModelAttrCollector(
    'lr_coef', Connector(processor=LogisticRegression, edges=y_edges), 'coef'
)
e_stacking.collect(coef)
coef.get_attrs_agg('lr20')

In [43]:
e_stacking.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending=False).iloc[:5]

,valid,train_sub
lr20,0.918643,0.918646
lr19,0.918643,0.918646
lr18,0.918569,0.918571
lr17,0.918340,0.918346
lr16,0.916845,0.916845


# Submission

In [53]:
e_skf5.add_trainer('trainer')
e_skf5.trainers['trainer'].select_head(['cb_4', 'cb_5', 'lgb_3', 'xgb_4', 'xgb_6', 'lgb_5', 'nn_7', 'lgb_8', 'xgb_9'])
e_skf5.trainers['trainer'].train()

nn_6 5/5 (100%)                                                         
Train complete: 5 node(s)


In [54]:
e_stacking.add_trainer('trainer')
e_stacking.trainers['trainer'].select_head(['lr19'])
e_stacking.trainers['trainer'].train()

lr19 1/1 (100%)                 
Train complete: 1 node(s)


In [56]:
df_result = e_stacking.trainers['trainer'].to_inferencer(slice(-1, None)).process(
    e_skf5.trainers['trainer'].to_inferencer(slice(-1, None)).process(df_test)
)[['lr19__Churn_1']]
df_result.columns = ['Churn']

In [57]:
df_result.with_columns(
    df_test['id']
)[['id', 'Churn']].write_csv('data/submission.csv')

In [58]:
# !kaggle competitions submit -c playground-series-s6e3 -f data/submission.csv -m "9"

100%|██████████████████████████████████████| 6.51M/6.51M [00:05<00:00, 1.26MB/s]
Successfully submitted to Predict Customer Churn